# Time-Series Forecasting of Carbon Monoxide and Nitrogen Dioxide Levels

Build predictive models to forecast daily or hourly concentrations of **CO(GT)** and **NO₂(GT)** based on historical data.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA as _ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(42)
    KERAS_AVAILABLE = True
    print(f"TensorFlow available: {tf.__version__}")
except ImportError:
    KERAS_AVAILABLE = False
    print("TensorFlow not available — LSTM section will be skipped.")

plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})
sns.set_style('darkgrid')
np.random.seed(42)
print("All core libraries loaded.")

In [ ]:
url = 'https://raw.githubusercontent.com/rashakil-ds/Public-Datasets/refs/heads/main/airquality.csv'
df_raw = pd.read_csv(url)

# Combine Date & Time into a single datetime index
df_raw['Datetime'] = pd.to_datetime(
    df_raw['Date'].astype(str) + ' ' + df_raw['Time'].astype(str)
)
df_raw.set_index('Datetime', inplace=True)
df_raw.drop(['Date', 'Time'], axis=1, inplace=True)

# Replace -200 sentinel values (dataset encoding for missing readings) with NaN
df_raw.replace(-200, np.nan, inplace=True)

print(f"Shape:      {df_raw.shape}")
print(f"Date range: {df_raw.index.min()} → {df_raw.index.max()}")
df_raw.head()

In [ ]:
print("Missing values per column:")
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

# Project Workflow

### **Deliverables**:
1. **Forecasting Models**:
   - Trained models for `CO(GT)` and `NO₂(GT)`.
2. **Forecast Visualization**:
   - Plots showing historical trends and future predictions.
3. **Evaluation Report**:
   - Metrics and comparison of different forecasting models.
4. **Insights and Recommendations**:
   - Suggestions for mitigating high levels of these gases based on predictions.


### 1. Data Preprocessing
- **Date-Time Parsing**:
  - Combine the `Date` and `Time` columns into a single `datetime` column.
  - Set the `datetime` column as the index of the dataset.
- **Resampling**:
  - Aggregate the data into meaningful time intervals (e.g., hourly or daily averages).
- **Handle Missing Values**:
  - Use interpolation, mean, or advanced imputation techniques to fill missing data for `CO(GT)` and `NO₂(GT)`.
- **Outlier Detection**:
  - Remove or cap extreme values in `CO(GT)` and `NO₂(GT)` using statistical thresholds.

In [ ]:
# Resample to daily averages — reduces sensor noise, simplifies forecasting
df = df_raw.resample('D').mean()

# Linear interpolation for interior gaps; forward/back fill for edges
df.interpolate(method='linear', limit_direction='both', inplace=True)
df.ffill(inplace=True)
df.bfill(inplace=True)

assert df.isnull().sum().sum() == 0, "Missing values remain after imputation!"

TARGETS = ['CO(GT)', 'NO2(GT)']
print(f"Resampled to daily: {df.shape[0]} days × {df.shape[1]} features")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
df[TARGETS].describe().round(3)

In [ ]:
def cap_iqr(series, k=3.0):
    """Winsorize outliers using k × IQR bounds."""
    Q1, Q3 = series.quantile([0.25, 0.75])
    lo, hi = Q1 - k * (Q3 - Q1), Q3 + k * (Q3 - Q1)
    return series.clip(lo, hi), lo, hi

for col in TARGETS:
    df[col], lo, hi = cap_iqr(df[col])
    print(f"{col:12s} → clipped to [{lo:.3f}, {hi:.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, TARGETS):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='lightsteelblue'))
    ax.set_title(f'{col}  (post-capping)')
    ax.set_ylabel(col)
plt.suptitle('Outlier Removal — Box Plots', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("Cleaned Dataset Summary")
print("=" * 45)
for col in TARGETS:
    s = df[col]
    print(f"\n  {col}")
    print(f"    Mean ± Std : {s.mean():.3f} ± {s.std():.3f}")
    print(f"    Min / Max  : {s.min():.3f} / {s.max():.3f}")
    print(f"    Skewness   : {s.skew():.3f}")

### 2. Exploratory Data Analysis (EDA)
- **Trend Analysis**:
  - Visualize the long-term trends of `CO(GT)` and `NO₂(GT)`.
- **Seasonality**:
  - Identify seasonal patterns (e.g., daily or yearly fluctuations).
- **Correlation Analysis**:
  - Explore relationships between `CO(GT)`, `NO₂(GT)`, and other features (e.g., temperature or humidity).


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
colors = ['steelblue', 'darkorange']

for ax, col, color in zip(axes, TARGETS, colors):
    ax.plot(df.index, df[col], color=color, lw=1.2, alpha=0.8, label='Daily')
    ax.fill_between(df.index, df[col], alpha=0.15, color=color)
    ma7 = df[col].rolling(7, center=True).mean()
    ax.plot(df.index, ma7, 'k--', lw=2, label='7-day MA')
    ax.set_ylabel(col, fontsize=12)
    ax.legend(loc='upper right')

axes[0].set_title('Daily CO(GT) and NO₂(GT) Concentrations (2004–2005)',
                  fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
df_eda = df[TARGETS].copy()
df_eda['Month']     = df_eda.index.month
df_eda['DayOfWeek'] = df_eda.index.dayofweek

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dow_labels   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
colors       = ['steelblue', 'darkorange']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (col, color) in enumerate(zip(TARGETS, colors)):
    # Monthly distribution
    sns.boxplot(data=df_eda, x='Month', y=col, ax=axes[i][0],
                color=color, fliersize=2)
    axes[i][0].set_xticklabels(
        [month_labels[m - 1] for m in sorted(df_eda['Month'].unique())]
    )
    axes[i][0].set_title(f'{col} — Monthly Distribution')
    axes[i][0].set_xlabel('')

    # Day-of-week distribution
    sns.boxplot(data=df_eda, x='DayOfWeek', y=col, ax=axes[i][1],
                color=color, fliersize=2)
    axes[i][1].set_xticklabels(dow_labels)
    axes[i][1].set_title(f'{col} — Day-of-Week Distribution')
    axes[i][1].set_xlabel('')

plt.suptitle('Seasonality Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Full feature correlation heatmap
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], cbar_kws={'shrink': 0.8}, linewidths=0.4)
axes[0].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# CO vs NO₂ scatter
r = df['CO(GT)'].corr(df['NO2(GT)'])
axes[1].scatter(df['CO(GT)'], df['NO2(GT)'], alpha=0.45, s=18, color='purple')
axes[1].set_xlabel('CO(GT)')
axes[1].set_ylabel('NO₂(GT)')
axes[1].set_title(f'CO(GT) vs NO₂(GT)  (Pearson r = {r:.2f})',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


### 3. Feature Engineering
- Create lag features for `CO(GT)` and `NO₂(GT)` to incorporate historical values.
- Add time-based features (e.g., hour of the day, day of the week, month).
- Include environmental factors (`T`, `RH`, `AH`) as predictors.


In [ ]:
def build_features(df, targets, lag_steps=(1, 2, 3, 7, 14), roll_windows=(3, 7)):
    """
    Construct lag, rolling-statistics, and cyclical time features.
    All lags use .shift(n) to prevent data leakage.
    """
    feat = df.copy()
    for col in targets:
        for lag in lag_steps:
            feat[f'{col}_lag{lag}'] = feat[col].shift(lag)
        for w in roll_windows:
            base = feat[col].shift(1)          # shift to avoid leakage
            feat[f'{col}_roll{w}_mean'] = base.rolling(w).mean()
            feat[f'{col}_roll{w}_std']  = base.rolling(w).std()

    # Raw time indices
    feat['month']     = feat.index.month
    feat['dayofweek'] = feat.index.dayofweek
    feat['dayofyear'] = feat.index.dayofyear
    feat['quarter']   = feat.index.quarter

    # Cyclical encoding — avoids ordinal bias (e.g. Dec ≈ Jan)
    feat['month_sin'] = np.sin(2 * np.pi * feat['month']     / 12)
    feat['month_cos'] = np.cos(2 * np.pi * feat['month']     / 12)
    feat['dow_sin']   = np.sin(2 * np.pi * feat['dayofweek'] / 7)
    feat['dow_cos']   = np.cos(2 * np.pi * feat['dayofweek'] / 7)

    feat.dropna(inplace=True)
    return feat

df_feat = build_features(df, TARGETS)
feature_cols = [c for c in df_feat.columns if c not in TARGETS]
print(f"Feature matrix: {df_feat.shape[0]} rows × {len(feature_cols)} features  (+{len(TARGETS)} targets)")
df_feat.head()

In [ ]:
# Time-based split: 70% train / 15% val / 15% test  (no shuffling — respects temporal order)
n      = len(df_feat)
i_val  = int(n * 0.70)
i_test = int(n * 0.85)

train = df_feat.iloc[:i_val]
val   = df_feat.iloc[i_val:i_test]
test  = df_feat.iloc[i_test:]

print(f"Train : {train.index.min().date()} → {train.index.max().date()}  ({len(train):>3d} days)")
print(f"Val   : {val.index.min().date()}   → {val.index.max().date()}    ({len(val):>3d} days)")
print(f"Test  : {test.index.min().date()}  → {test.index.max().date()}   ({len(test):>3d} days)")

X_train, y_train = train[feature_cols], train[TARGETS]
X_val,   y_val   = val[feature_cols],   val[TARGETS]
X_test,  y_test  = test[feature_cols],  test[TARGETS]

# Boundary for ARIMA (uses original df, not df_feat)
arima_cutoff     = train.index.max()
arima_test_index = test.index

In [ ]:
print(f"Total features ({len(feature_cols)}):")
for i, fc in enumerate(feature_cols, 1):
    print(f"  {i:>2}. {fc}")
print()
X_train.describe().round(3)

### 4. Time-Series Forecasting
- Train separate models for **CO(GT)** and **NO₂(GT)**.
- Explore the following forecasting approaches:
  - **Statistical Models**:
    - Classical models like ARIMA or SARIMA for univariate forecasting.
  - **Machine Learning Models**:
    - Use regression models trained on lag features and external predictors.
  - **Deep Learning Models**:
    - Apply LSTM, GRU, or other RNN-based models for sequence forecasting.
- Evaluate and compare the models.


In [ ]:
arima_results = {}

for col in TARGETS:
    series_train = df.loc[df.index <= arima_cutoff, col]
    series_test  = df.loc[arima_test_index, col]

    # ADF test on first-differenced series to confirm I(1) assumption
    adf_pval = adfuller(series_train.diff().dropna())[1]
    status   = "stationary" if adf_pval < 0.05 else "may need d>1"
    print(f"{col}: ADF p-value (1st diff) = {adf_pval:.4f}  [{status}]")

    model  = _ARIMA(series_train, order=(2, 1, 2))
    result = model.fit()

    forecast = result.forecast(steps=len(series_test))

    arima_results[col] = {
        'actual':   series_test.values,
        'forecast': np.clip(forecast.values, 0, None),   # concentrations ≥ 0
    }
    print(f"  AIC = {result.aic:.2f}  |  test samples = {len(series_test)}\n")

print("ARIMA models fitted successfully.")

In [ ]:
xgb_models  = {}
xgb_results = {}

XGB_PARAMS = dict(
    n_estimators     = 600,
    learning_rate    = 0.04,
    max_depth        = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    gamma            = 0.1,
    random_state     = 42,
    n_jobs           = -1,
)

for col in TARGETS:
    model = xgb.XGBRegressor(
        **XGB_PARAMS,
        early_stopping_rounds = 30,
        eval_metric           = 'mae',
    )
    model.fit(
        X_train, y_train[col],
        eval_set = [(X_val, y_val[col])],
        verbose  = False,
    )
    preds = np.clip(model.predict(X_test), 0, None)

    xgb_models[col]  = model
    xgb_results[col] = {'actual': y_test[col].values, 'forecast': preds}

    fi = pd.Series(model.feature_importances_, index=feature_cols).nlargest(10)
    print(f"\n{'─'*40}")
    print(f"XGBoost — {col}  (best iter: {model.best_iteration})")
    print(fi.to_string())

print("\nXGBoost models fitted successfully.")

In [ ]:
LOOKBACK     = 14     # 2-week context window
lstm_results = {}

def make_sequences(X, y, lookback=LOOKBACK):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

if KERAS_AVAILABLE:
    scaler_X  = MinMaxScaler().fit(X_train.values)
    X_all_s   = scaler_X.transform(df_feat[feature_cols].values)
    seq_dates = df_feat.index[LOOKBACK:]   # date label for each sequence

    for col in TARGETS:
        scaler_y = MinMaxScaler().fit(y_train[[col]].values)
        y_all_s  = scaler_y.transform(df_feat[col].values.reshape(-1, 1)).ravel()

        X_seq, y_seq = make_sequences(X_all_s, y_all_s)

        is_tr = seq_dates < val.index[0]
        is_va = (seq_dates >= val.index[0]) & (seq_dates < test.index[0])
        is_te = seq_dates >= test.index[0]

        X_tr, y_tr = X_seq[is_tr], y_seq[is_tr]
        X_va, y_va = X_seq[is_va], y_seq[is_va]
        X_te, y_te = X_seq[is_te], y_seq[is_te]

        model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(LOOKBACK, X_tr.shape[2])),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1),
        ])
        model.compile(optimizer='adam', loss='mae')

        es = EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')
        hist = model.fit(
            X_tr, y_tr,
            validation_data = (X_va, y_va),
            epochs=100, batch_size=16,
            callbacks=[es], verbose=0,
        )

        preds_s = model.predict(X_te, verbose=0).ravel()
        preds   = scaler_y.inverse_transform(preds_s.reshape(-1, 1)).ravel()
        actual  = scaler_y.inverse_transform(y_te.reshape(-1, 1)).ravel()

        lstm_results[col] = {
            'actual':   actual,
            'forecast': np.clip(preds, 0, None),
        }
        print(f"LSTM({col}): {len(hist.epoch)} epochs  |  test samples = {len(preds)}")
else:
    print("Skipping LSTM — TensorFlow not available.")

### 5. Model Evaluation
- Use appropriate metrics for forecasting:
  - Mean Absolute Error (MAE)
  - Root Mean Squared Error (RMSE)
  - Mean Absolute Percentage Error (MAPE)
- Plot actual vs. predicted values to visualize performance.


In [ ]:
def compute_metrics(actual, pred, model_name, target_name):
    actual, pred = np.array(actual), np.array(pred)
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / np.where(actual == 0, 1e-8, actual))) * 100
    return {'Model': model_name, 'Target': target_name,
            'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape}

rows = []
for col in TARGETS:
    rows.append(compute_metrics(arima_results[col]['actual'],
                                arima_results[col]['forecast'], 'ARIMA', col))
    rows.append(compute_metrics(xgb_results[col]['actual'],
                                xgb_results[col]['forecast'],   'XGBoost', col))
    if KERAS_AVAILABLE and lstm_results:
        rows.append(compute_metrics(lstm_results[col]['actual'],
                                    lstm_results[col]['forecast'], 'LSTM', col))

metrics_df = (pd.DataFrame(rows)
                .set_index(['Model', 'Target'])
                .round(4))
print("=== Model Evaluation Metrics (Test Set) ===")
metrics_df

In [ ]:
model_list = [('ARIMA',   arima_results, 'steelblue'),
              ('XGBoost', xgb_results,   'darkorange')]
if KERAS_AVAILABLE and lstm_results:
    model_list.append(('LSTM', lstm_results, 'seagreen'))

ncols = len(model_list)
fig, axes = plt.subplots(len(TARGETS), ncols, figsize=(6 * ncols, 9))

for row, col in enumerate(TARGETS):
    for c_idx, (mname, mres, color) in enumerate(model_list):
        ax  = axes[row][c_idx]
        act = np.array(mres[col]['actual'])
        prd = np.array(mres[col]['forecast'])
        n   = min(len(act), len(prd))

        ax.plot(act[:n], label='Actual', color='black', lw=1.8)
        ax.plot(prd[:n], label=mname,    color=color,   lw=1.8, ls='--')
        ax.set_title(f'{col} — {mname}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xlabel('Test Day Index')
        if c_idx == 0:
            ax.set_ylabel(col)

plt.suptitle('Actual vs Predicted — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Best Model per Target (by MAE) ===\n")
for col in TARGETS:
    sub  = metrics_df.xs(col, level='Target')
    best = sub['MAE'].idxmin()
    print(f"  {col}: Best model → {best}")
    display(sub)
    print()

# Side-by-side bar chart
fig, axes = plt.subplots(1, len(TARGETS), figsize=(14, 5))
for ax, col in zip(axes, TARGETS):
    sub = metrics_df.xs(col, level='Target').reset_index()
    sub.set_index('Model')[['MAE', 'RMSE']].plot(
        kind='bar', ax=ax, rot=0,
        color=['#4C72B0', '#DD8452'], edgecolor='black', width=0.6
    )
    ax.set_title(f'{col} — Error Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel('Error Value')
    ax.legend()

plt.suptitle('MAE & RMSE Across Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Visualization and Insights
- Generate forecasts for future dates and visualize the trends.
- Highlight seasonal patterns, peak times, and periods of concern.
- Provide actionable recommendations based on predicted trends.



In [ ]:
FORECAST_DAYS = 30

def rolling_forecast_xgb(model, df_feat, feature_cols, target_col, df_orig, steps=30):
    """
    Iterative multi-step forecast:  predict → append to history → re-engineer lags → repeat.
    Environmental features (T, RH, AH, sensors) are carried forward from the last known value.
    """
    history   = df_feat.copy()
    hist_orig = df_orig[[target_col]].copy()
    preds     = []

    lag_steps    = [1, 2, 3, 7, 14]
    roll_windows = [3, 7]

    for _ in range(steps):
        last_date = history.index[-1]
        next_date = last_date + pd.Timedelta(days=1)
        row = {}

        # Lag features from rolling history
        for lag in lag_steps:
            key = f'{target_col}_lag{lag}'
            row[key] = hist_orig[target_col].iloc[-lag] if lag <= len(hist_orig) else hist_orig[target_col].iloc[0]

        # Rolling statistics
        for w in roll_windows:
            vals = hist_orig[target_col].iloc[-w:]
            row[f'{target_col}_roll{w}_mean'] = vals.mean()
            row[f'{target_col}_roll{w}_std']  = vals.std()

        # Carry forward all other features (sensor/environmental readings)
        for c in feature_cols:
            if c not in row:
                row[c] = history[c].iloc[-1]

        # Update time features for next_date
        row.update({
            'month':     next_date.month,
            'dayofweek': next_date.dayofweek,
            'dayofyear': next_date.dayofyear,
            'quarter':   next_date.quarter,
            'month_sin': np.sin(2 * np.pi * next_date.month     / 12),
            'month_cos': np.cos(2 * np.pi * next_date.month     / 12),
            'dow_sin':   np.sin(2 * np.pi * next_date.dayofweek / 7),
            'dow_cos':   np.cos(2 * np.pi * next_date.dayofweek / 7),
        })

        X_next = pd.DataFrame([row], index=[next_date])[feature_cols]
        pred   = float(np.clip(model.predict(X_next)[0], 0, None))
        preds.append(pred)

        # Append to rolling histories
        new_row = X_next.copy()
        new_row[target_col] = pred
        for other_t in TARGETS:
            if other_t != target_col and other_t not in new_row.columns:
                new_row[other_t] = history[other_t].iloc[-1]
        history = pd.concat([history, new_row])
        hist_orig = pd.concat([
            hist_orig,
            pd.DataFrame({target_col: [pred]}, index=[next_date])
        ])

    future_idx = pd.date_range(df_feat.index[-1] + pd.Timedelta(days=1), periods=steps)
    return pd.Series(preds, index=future_idx, name=target_col)


future_preds = {}
for col in TARGETS:
    future_preds[col] = rolling_forecast_xgb(
        xgb_models[col], df_feat, feature_cols, col, df, FORECAST_DAYS
    )
    print(f"Generated {FORECAST_DAYS}-day forecast for  {col}")

In [ ]:
HIST_DAYS = 90   # show last 90 historical days for context

fig, axes = plt.subplots(len(TARGETS), 1, figsize=(14, 9))
colors = ['steelblue', 'darkorange']

for ax, col, color in zip(axes, TARGETS, colors):
    hist = df[col].iloc[-HIST_DAYS:]
    fp   = future_preds[col]

    ax.plot(hist.index, hist.values, color=color, lw=1.8, label='Historical')
    ax.plot(fp.index,   fp.values,   color='crimson', lw=2, ls='--', label='Forecast')
    ax.fill_between(fp.index, fp.values * 0.88, fp.values * 1.12,
                    alpha=0.18, color='crimson', label='±12% uncertainty band')

    ax.axvline(df.index[-1], color='gray', ls=':', lw=1.5, label='Forecast start')
    ax.set_ylabel(col, fontsize=12)
    ax.set_title(f'{col}: Last {HIST_DAYS} Days + {FORECAST_DAYS}-Day Ahead Forecast',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper left')

axes[-1].set_xlabel('Date')
plt.suptitle(f'{FORECAST_DAYS}-Day Forecasts via XGBoost (Best ML Model)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 65)
print("  FORECAST INSIGHTS")
print("=" * 65)

for col in TARGETS:
    fp        = future_preds[col]
    hist_mean = df[col].mean()
    hist_std  = df[col].std()
    fore_mean = fp.mean()
    fore_max  = fp.max()
    fore_peak = fp.idxmax().date()
    high_days = int((fp > hist_mean + hist_std).sum())
    trend     = "Rising" if fp.iloc[-7:].mean() > fp.iloc[:7].mean() else "Falling"

    print(f"\n  {col}")
    print(f"    Historical mean ± σ : {hist_mean:.3f} ± {hist_std:.3f}")
    print(f"    Forecast mean       : {fore_mean:.3f}  ({fore_mean - hist_mean:+.3f} vs historical)")
    print(f"    Forecast peak       : {fore_max:.3f}  on {fore_peak}")
    print(f"    Late-period trend   : {trend}")
    print(f"    High-risk days (>μ+σ): {high_days} / {FORECAST_DAYS}")

print(f"\n{'='*65}")
print("  ACTION RECOMMENDATIONS")
print(f"{'─'*65}")
recommendations = [
    "1. Issue public health alerts on forecasted high-concentration days.",
    "2. Enforce stricter traffic/industrial emission controls during predicted peaks.",
    "3. Cross-validate forecasts with meteorological data (wind, precipitation) for robustness.",
    "4. Retrain models monthly to capture seasonal drift and sensor degradation.",
    "5. Apply SHAP analysis on XGBoost to quantify the top causal drivers per target.",
    "6. Automate threshold alerts based on WHO air quality limits:",
    "      CO  → 4 mg/m³ (8-hour mean)",
    "      NO₂ → 25 μg/m³ (24-hour mean)",
    "7. Extend pipeline to hourly forecasting for intraday emission management.",
]
for r in recommendations:
    print(f"  {r}")
print("=" * 65)